# التحقق من الصحة: التوجيه الثابت بدون إطلاق النار (التعيين القسري عبر البيانات الوصفية)

## نظرة عامة
في هذا الدفتري، نستفيد من ميزة `allowed_mask` الخاصة بـ `Router` للتحقق من صحة آلية "توجيه اللقطة الصفرية" التي تفرض 100% من حركة المرور نحو مشبك تشابكي محدد (على سبيل المثال، مشبك الآلة الحاسبة) استنادًا إلى البيانات التعريفية لبيانات الإدخال.
نؤكد أنه، دون تدريب جهاز التوجيه على الإطلاق، يمكننا إرسال رموز محددة بشكل موثوق إلى Synapse المقابل.

In [ ]:
import os
import sys

# Setup for running on Colab
if 'google.colab' in sys.modules:
    !git clone https://github.com/JunSuzukiJapan/SynapticRouter.git
    %cd SynapticRouter

sys.path.append('.')
sys.path.append('./src')
if 'google.colab' not in sys.modules:
    sys.path.append('..')
    sys.path.append('../src')

import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from sra_reference import SRAModel, CalculatorSynapse
from constants import PAD

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# 1. Model initialization
VOCAB_SIZE = 128
DIM = 64
LAYERS = 2
NUM_SYNAPSES = 4
K = 2
SYN_HIDDEN = 128

model = SRAModel(
    vocab_size=VOCAB_SIZE, 
    dim=DIM, 
    layers=LAYERS, 
    num_synapses=NUM_SYNAPSES, 
    k=K, 
    syn_hidden=SYN_HIDDEN
).to(device)

print(f"Initial number of Synapses: {model.blocks[0].router.num_synapses}")

In [ ]:
# 2. Add a CalculatorSynapse
# Add a Synapse that performs deterministic arithmetic
def calc_factory():
    return CalculatorSynapse(dim=DIM)

def emb_factory():
    # Start from a random initial Embedding
    return torch.randn(DIM)

model.add_custom_synapse(calc_factory, emb_factory)
model = model.to(device)

calc_synapse_idx = NUM_SYNAPSES # Index of the newly added Synapse (equal to the original count)
print(f"Number of Synapses after addition: {model.blocks[0].router.num_synapses}")
print(f"Index of CalculatorSynapse: {calc_synapse_idx}")

In [ ]:
# 3. Build dummy data
# Assume an input sequence with B=1, T=10
BATCH_SIZE = 1
SEQ_LEN = 10
x = torch.randint(1, VOCAB_SIZE, (BATCH_SIZE, SEQ_LEN)).to(device)
y_in = torch.randint(1, VOCAB_SIZE, (BATCH_SIZE, SEQ_LEN)).to(device)

print(f"Input sequence (x): {x.shape}")

In [ ]:
# 4. Zero-Shot routing (behavior without allowed_mask)
model.eval()
with torch.no_grad():
    _, router_logits_normal, _ = model(x, y_in)
    
# Check the routing weights of the last block
last_block_logits = router_logits_normal[-1] # (B, T_total, num_synapses)
# Keep only the target positions (T tokens after the input)
last_block_logits_tgt = last_block_logits[:, x.size(1):, :]
weights_normal = torch.softmax(last_block_logits_tgt, dim=-1)

print("Routing probabilities without allowed_mask (first token):")
for i in range(model.blocks[0].router.num_synapses):
    print(f"  Synapse {i}: {weights_normal[0, 0, i].item():.4f}")

In [ ]:
# 5. Forced routing via allowed_mask
# Force only the first 3 tokens to be routed to the CalculatorSynapse
total_seq_len = x.size(1) + y_in.size(1)
allowed_mask = torch.ones((BATCH_SIZE, total_seq_len, model.blocks[0].router.num_synapses), dtype=torch.bool).to(device)

# Suppose (metadata) tells us that the first 3 target positions are a calculation task
# The T_target indices start from x.size(1)
target_start = x.size(1)

# Initially all Synapses are allowed, but for the specified tokens we set every Synapse except CalculatorSynapse to False
for t in range(3):
    idx = target_start + t
    allowed_mask[0, idx, :] = False
    allowed_mask[0, idx, calc_synapse_idx] = True

print("Pass allowed_mask to SRAModel.forward for validation.")

In [ ]:
# 6. End-to-end validation on the SRAModel
model.eval()
with torch.no_grad():
    # 1. Routing without a mask (repeated)
    _, router_logits_nomask, _ = model(x, y_in)
    logits_nomask = router_logits_nomask[-1][:, x.size(1):, :]
    
    # 2. Routing with a mask
    _, router_logits_mask, _ = model(x, y_in, allowed_mask=allowed_mask)
    logits_mask = router_logits_mask[-1][:, x.size(1):, :]

# Compute the probabilities of all Synapses from the logits
probs_nomask = torch.softmax(logits_nomask, dim=-1)
probs_mask = torch.softmax(logits_mask, dim=-1)

print("--- Without mask ---")
print("Routing probabilities for the first token:")
for i in range(model.blocks[0].router.num_synapses):
    print(f"  Synapse {i}: {probs_nomask[0, 0, i].item():.4f}")

print("\n--- With mask (Calculator forced) ---")
print("Routing probabilities for the first token:")
for i in range(model.blocks[0].router.num_synapses):
    print(f"  Synapse {i}: {probs_mask[0, 0, i].item():.4f}")

# Check the 4th token (not forced)
print("\n--- With mask (4th token, not forced) ---")
print("Routing probabilities:")
for i in range(model.blocks[0].router.num_synapses):
    print(f"  Synapse {i}: {probs_mask[0, 3, i].item():.4f}")

## مناقشة
من نتائج التحقق من الصحة، أكدنا أن استخدام `allowed_mask` يتيح لنا توجيه حركة المرور إلى Synapse المحدد (CalculatorSynapse في هذه الحالة) باحتمال 100%، دون أي ضبط أو تدريب مسبق على الإطلاق.

يوضح هذا جدوى "Zero-Shot Hard Routing"، حيث يمكن لمصنف خارجي أو تعبير عادي أو بيانات وصفية سريعة أن يجبر النموذج على استخدام الحساب الحتمي أو الاسترجاع الواقعي (VectorDB).

بفضل التغييرات التي تم إجراؤها على البنية الأساسية (`SRAModel`)، أصبح التحكم في المجال ممكنًا الآن في وقت الاستدلال ببساطة عن طريق تمرير قناع ديناميكيًا.